In [1]:
%%writefile bubble.cpp

#include <iostream>
#include <omp.h>

using namespace std;

void bubbleSort(int arr[], int n)
{
    for(int i = 0; i < n - 1; i++)
    {
        for(int j = 0; j < n - i - 1; j++)
        {
            if(arr[j] > arr[j + 1])
            {
                swap(arr[j], arr[j + 1]);
            }
        }
    }
}

// Parallel Bubble Sort using OpenMP
void parallelBubbleSort(int arr[], int n)
{
    #pragma omp parallel
    {
        for(int i = 0; i < n; i++)
        {
            // Odd phase
            #pragma omp for
            for(int j = 1; j < n; j += 2)
            {
                if(arr[j] < arr[j - 1])
                {
                    swap(arr[j], arr[j - 1]);
                }
            }

            // Even phase
            #pragma omp for
            for(int j = 2; j < n; j += 2)
            {
                if(arr[j] < arr[j - 1])
                {
                    swap(arr[j], arr[j - 1]);
                }
            }
        }
    }
}


void merge(int arr[], int low, int mid, int high)
{
    int n1 = mid - low + 1;
    int n2 = high - mid;

    int left[n1];
    int right[n2];

    for(int i = 0; i < n1; i++)
        left[i] = arr[low + i];

    for(int j = 0; j < n2; j++)
        right[j] = arr[mid + 1 + j];

    int i = 0, j = 0, k = low;

    while(i < n1 && j < n2)
    {
        if(left[i] <= right[j])
        {
            arr[k] = left[i];
            i++;
        }
        else
        {
            arr[k] = right[j];
            j++;
        }
        k++;
    }

    while(i < n1)
    {
        arr[k] = left[i];
        i++;
        k++;
    }

    while(j < n2)
    {
        arr[k] = right[j];
        j++;
        k++;
    }
}

void mergeSort(int arr[], int low, int high)
{
    if(low < high)
    {
        int mid = (low + high) / 2;

        mergeSort(arr, low, mid);
        mergeSort(arr, mid + 1, high);

        merge(arr, low, mid, high);
    }
}

// Parallel Merge Sort
void parallelMergeSort(int arr[], int low, int high)
{
    if(low < high)
    {
        int mid = (low + high) / 2;

        #pragma omp parallel sections
        {
            #pragma omp section
            {
                parallelMergeSort(arr, low, mid);
            }

            #pragma omp section
            {
                parallelMergeSort(arr, mid + 1, high);
            }
        }

        merge(arr, low, mid, high);
    }
}



void printArray(int arr[], int n)
{
    for(int i = 0; i < n; i++)
    {
        cout << arr[i] << " ";
    }
    cout << endl;
}



int main()
{
    int n;

    cout << "Enter number of elements: ";
    cin >> n;

    int arr1[n], arr2[n], arr3[n], arr4[n];

    cout << "Enter elements:\n";

    for(int i = 0; i < n; i++)
    {
        cin >> arr1[i];

        // Copy data into all arrays
        arr2[i] = arr1[i];
        arr3[i] = arr1[i];
        arr4[i] = arr1[i];
    }

    double start, end;


    cout << "\nOriginal Array:\n";
    printArray(arr1, n);

    start = omp_get_wtime();
    bubbleSort(arr1, n);
    end = omp_get_wtime();

    cout << "\nSequential Bubble Sort:\n";
    printArray(arr1, n);

    cout << "Time Taken: " << end - start << " seconds\n";


    start = omp_get_wtime();
    parallelBubbleSort(arr2, n);
    end = omp_get_wtime();

    cout << "\nParallel Bubble Sort:\n";
    printArray(arr2, n);

    cout << "Time Taken: " << end - start << " seconds\n";


    start = omp_get_wtime();
    mergeSort(arr3, 0, n - 1);
    end = omp_get_wtime();

    cout << "\nSequential Merge Sort:\n";
    printArray(arr3, n);

    cout << "Time Taken: " << end - start << " seconds\n";



    start = omp_get_wtime();
    parallelMergeSort(arr4, 0, n - 1);
    end = omp_get_wtime();

    cout << "\nParallel Merge Sort:\n";
    printArray(arr4, n);

    cout << "Time Taken: " << end - start << " seconds\n";

    return 0;
}



Writing bubble.cpp


In [2]:
!g++ bubble.cpp -fopenmp -o bubble
!./bubble

Enter number of elements: 10
Enter elements:
5
20
25
40
15
69
72
35
45
42

Original Array:
5 20 25 40 15 69 72 35 45 42 

Sequential Bubble Sort:
5 15 20 25 35 40 42 45 69 72 
Time Taken: 1.3e-06 seconds

Parallel Bubble Sort:
5 15 20 25 35 40 42 45 69 72 
Time Taken: 0.000111088 seconds

Sequential Merge Sort:
5 15 20 25 35 40 42 45 69 72 
Time Taken: 1.433e-06 seconds

Parallel Merge Sort:
5 15 20 25 35 40 42 45 69 72 
Time Taken: 3.954e-05 seconds
